## ToyAIKit

The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. It does the same thing as our handwritten loop with less boilerplate. If you open its runners code, you'll find the same while True loop we wrote by hand.

I use it here on purpose, because I don't want to pick a winner among the production frameworks. ToyAIKit is small and easy to read, so when something breaks you can see exactly what happened. That makes it handy for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. We use it because it's minimal and you can see what it does.

Install it:

In [1]:
!uv add toyaikit

Resolved 129 packages in 1ms
Checked 125 packages in 1ms


Import the classes we need:

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

Repeat the definitions of `search()`, `search_tool`, and `index` for local use...

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

If we add a type hint and a docstring to `search()`, `ToyAIKit` reads them and derives the schema for us:

In [8]:
def search(query: str) -> dict[str, str]:   # return type hint: 
                                            # search() is expected to return a dictionary 
                                            # with string keys and string values
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Now, register it without passing a schema:

In [9]:
agent_tools = Tools()
agent_tools.add_tool(search)

Now take a look at what Tools() produced...

In [7]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we hand-wrote in the function calling lesson. `ToyAIKit` generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The `OpenAI Agents SDK`, `PydanticAI`, `LangChain` and `Google ADK` all work this way. You write the tool and the framework figures out how to describe it.

**Note that the return type hint for search() wasn't used by toyAIKit.tools.**

toyAIKit.tools can use query: str to construct the tool’s input schema, because the language model needs to know what arguments to supply. The return annotation is usually irrelevant to the tool schema: the framework simply calls the function and passes whatever value it actually returns back to the model.

### The chat interface and runner
Create the chat interface and a callback, then build the runner:

In [11]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [12]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The `chat_interface` handles display in the notebook. The `callback` renders model messages and tool calls as they happen. The `runner` runs the agent loop, the same `while True` we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

We pick `gpt-5.4-mini` here on purpose. Without it, `ToyAIKit` falls back to a smaller, faster default that doesn't follow the instructions as reliably.

Run a single prompt:

In [15]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [16]:
type(result)  # <class 'toyaikit.chat.runners.OpenAIResponsesRunnerResult'>

toyaikit.chat.runners.LoopResult

We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". The recovery is the same as the handwritten loop. The notebook output is nicer to watch. Each tool call and message renders inline, so you can look at every search result.

The result is a `LoopResult` with all_messages (the full conversation), token counts, and cost (computed from token usage).

Look at what the call cost:

In [17]:
result.cost

CostInfo(input_cost=Decimal('0.003426'), output_cost=Decimal('0.001449'), total_cost=Decimal('0.004875'))

Look at the full message history:

In [21]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally local Ollama how do I run locally"}', call_i

### Continuing the conversation
Take the messages from the previous result and pass them as previous_messages on the next loop call:

In [22]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


### Interactive chat

For a chat-like workflow, run the built-in input loop:

In [25]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"course logistics FAQ office hours grading homework exam deadlines s

Type questions and get answers. Type "stop" to exit.